In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("1. 正在获取最新鲜的数据 (保留原始日期)...")
data = yf.download("SPY", period="2y", interval="1d", progress=False)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = [col[0] for col in data.columns]
data['Close'] = pd.to_numeric(data['Close'], errors='coerce')

print("2. 计算 50日均线、20日动量 与 20日波动率...")
data['MA50'] = data['Close'].rolling(window=50, min_periods=50).mean()
data['momentum_20'] = data['Close'] / data['Close'].shift(20) - 1

# 新增：计算 20 日年化波动率（标准差 * 根号252）
data['Daily_Return'] = data['Close'].pct_change()
data['Volatility_20'] = data['Daily_Return'].rolling(window=20).std() * np.sqrt(252)

def mad_filter(series, n=3):
    median = series.median()
    mad = (series - median).abs().median()
    upper_bound = median + n * mad * 1.4826
    lower_bound = median - n * mad * 1.4826
    return series.clip(lower=lower_bound, upper=upper_bound)

data['momentum_20'] = mad_filter(data['momentum_20'])
data = data.dropna().copy() 

print("3. 执行对冲基金级仓位管理：信心指数 + 波动率风控...")

# A. 风险评价 (Risk Parity)：设定目标年化波动率为 16% (0.16)
# 波动越大，算出来的仓位权重越小；上限截断为 1.0 (不加杠杆)
target_volatility = 0.16
data['Vol_Weight'] = (target_volatility / data['Volatility_20']).clip(upper=1.0)

# B. 信心指数 (Momentum Strength)：动量 0~2.5% 线性映射为 0~100% 的仓位
# 动量极强(>10%)则信心满格，动量微弱(1%)则只给 10% 的基础权重
data['Mom_Weight'] = (data['momentum_20'] / 0.025).clip(lower=0.0, upper=1.0)

# C. 合成最终信号：趋势过滤 + 信心乘数 + 风险乘数
data['Signal'] = 0.0 # 默认为空仓 (0)

# 过滤条件：只有在“收盘价站上50日均线”且“动量为正”的上升趋势中，才分配连续仓位
valid_trend = (data['momentum_20'] > 0) & (data['Close'] > data['MA50'])
data.loc[valid_trend, 'Signal'] = data['Mom_Weight'] * data['Vol_Weight']

# T+1 防前视偏差（因为现在是连续仓位，每天都在动态调仓，所以不需要旧版的 ffill 缓冲区了）
data['Position'] = data['Signal'].shift(1).fillna(0)

print("4. 结算连续仓位收益率与核心指标...")
data['Market_Return'] = data['Close'].pct_change()
# 策略收益 = 动态分配的仓位 (例如 0.82) * 大盘真实涨跌幅
data['Strategy_Return'] = data['Position'] * data['Market_Return']
data = data.dropna().copy()

data['Cumulative_Market'] = (1 + data['Market_Return']).cumprod()
data['Cumulative_Strategy'] = (1 + data['Strategy_Return']).cumprod()

total_days = len(data)
strategy_annual_return = (data['Cumulative_Strategy'].iloc[-1]) ** (252 / total_days) - 1
excess_return = data['Strategy_Return'] - ((1 + 0.02) ** (1/252) - 1)
sharpe_ratio = np.sqrt(252) * (excess_return.mean() / data['Strategy_Return'].std())
rolling_max = data['Cumulative_Strategy'].cummax()
max_drawdown = ((data['Cumulative_Strategy'] - rolling_max) / rolling_max).min()

print("\n" + "=" * 40)
print(f"📊 高级V2策略 年化收益率: {strategy_annual_return * 100:.2f}%")
print(f"📈 高级V2策略 夏普比率:   {sharpe_ratio:.2f}")
print(f"📉 高级V2策略 最大回撤:   {max_drawdown * 100:.2f}%")
print("=" * 40)

print("5. 绘制最终版对比图...")
plt.figure(figsize=(12, 6))
plt.plot(data.index, data['Cumulative_Market'], label='Benchmark (SPY Buy & Hold)', color='gray', alpha=0.7, linestyle='--')
plt.plot(data.index, data['Cumulative_Strategy'], label='V2 Strategy (Risk Parity + Mom Strength)', color='red', linewidth=2)
plt.title('Hedge Fund Level Strategy vs Benchmark', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return (Base = 1.0)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()